# ML: Forward-Looking Customer Churn Prediction

Trains a distributed Spark `GBTClassifier` on historical customer snapshots and
calibrates its probabilities with a held-out chronological window.

## Model contract
- Activity is every purchase of a known customer: store `SALE` receipts and
  online orders, up to the source as-of time (the latest Silver receipt, shared
  by every required ML producer). Returns and anonymous purchases (null
  `customer_id`) never count.
- A customer enters a snapshot only when they purchased in the feature window
  before it; never-active and long-lapsed customers are not scored.
- Features use only purchases strictly before each snapshot boundary.
- Labels describe inactivity (no store or online purchase) in the forward
  window beginning at that boundary.
- Purchases are aggregated to customer-days before they are paired with
  snapshots, and each day pairs only with the snapshots whose windows contain
  it, so the work grows with activity rather than customers x snapshots.
- Train, calibration, and test labels are partitioned by availability with a
  90-day forward-label embargo between partitions.
- Trailing inactivity/recency is not a model feature.
- Published probabilities are isotonic-calibrated; the notebook fails closed
  when calibration is not identifiable.
- Gold output is current inference only: one row per customer active in the
  feature window before the current boundary. The deprecated
  `is_churned_actual` compatibility column is always null.


In [ ]:
from datetime import timedelta
import os

import mlflow
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.regression import IsotonicRegression
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException


In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================


def get_env(var_name, default=None):
    return os.environ.get(var_name, default)


LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="ag")
GOLD_DB = get_env("GOLD_DB", default="au")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="churn_prediction")
RECEIPTS_TABLE = get_env("RECEIPTS_TABLE", default="fact_receipts")
ONLINE_ORDERS_TABLE = get_env(
    "ONLINE_ORDERS_TABLE", default="fact_online_order_headers"
)
CUSTOMERS_TABLE = get_env("CUSTOMERS_TABLE", default="dim_customers")
CHURN_PREDICTIONS_TABLE = get_env(
    "CHURN_PREDICTIONS_TABLE", default="churn_predictions"
)
CHURN_PREDICTIONS_TABLE_NAME = (
    f"{LAKEHOUSE_NAME}.{GOLD_DB}.{CHURN_PREDICTIONS_TABLE}"
)
SCHEDULE_CADENCE = get_env("SCHEDULE_CADENCE", default="weekly")

CHURN_WINDOW_DAYS = int(get_env("CHURN_WINDOW_DAYS", default="90"))
FEATURE_WINDOW_DAYS = int(get_env("FEATURE_WINDOW_DAYS", default="180"))
SNAPSHOT_INTERVAL_DAYS = int(
    get_env("SNAPSHOT_INTERVAL_DAYS", default="7")
)
TRAIN_FRACTION = float(get_env("TRAIN_FRACTION", default="0.6"))
CALIBRATION_FRACTION = float(
    get_env("CALIBRATION_FRACTION", default="0.2")
)

MAX_ITER = int(get_env("MAX_ITER", default="80"))
MAX_DEPTH = int(get_env("MAX_DEPTH", default="5"))
STEP_SIZE = float(get_env("STEP_SIZE", default="0.1"))
SUBSAMPLING_RATE = float(get_env("SUBSAMPLING_RATE", default="1.0"))
RANDOM_SEED = int(get_env("RANDOM_SEED", default="42"))

if min(CHURN_WINDOW_DAYS, FEATURE_WINDOW_DAYS, SNAPSHOT_INTERVAL_DAYS) < 1:
    raise ValueError("Churn, feature, and snapshot windows must be positive.")
if not 0.0 < TRAIN_FRACTION < 1.0:
    raise ValueError("TRAIN_FRACTION must be between 0 and 1.")
if not 0.0 < CALIBRATION_FRACTION < 1.0:
    raise ValueError("CALIBRATION_FRACTION must be between 0 and 1.")
if TRAIN_FRACTION + CALIBRATION_FRACTION >= 1.0:
    raise ValueError("Train and calibration fractions must leave a test window.")

print(f"Configuration: SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"Output table: {CHURN_PREDICTIONS_TABLE_NAME}")
print(f"Feature history: {FEATURE_WINDOW_DAYS} days before each snapshot")
print(f"Forward churn window: {CHURN_WINDOW_DAYS} days")
print(f"Historical snapshot cadence: {SNAPSHOT_INTERVAL_DAYS} days")

# Physical ML contract used by repository validation and the pre-write gate.
ML_SOURCE_TABLES = ('fact_receipts', 'fact_online_order_headers', 'dim_customers')
ML_OUTPUT_CONTRACTS = {'churn_predictions': [('customer_id', 'long', False),
                       ('churn_probability', 'double', False),
                       ('churn_prediction', 'int', False),
                       ('risk_category', 'string', False),
                       ('is_churned_actual', 'int', True),
                       ('prediction_date', 'timestamp', False),
                       ('generated_at', 'timestamp', False),
                       ('model_version', 'string', False),
                       ('churn_window_days', 'int', False),
                       ('model_run_id', 'string', False),
                       ('schema_version', 'string', False)]}


In [ ]:
# SHARED SOURCE (generated - do not edit): retail_setup/generation/job_locks.py
# Edit utility/src/retail_setup/generation/job_locks.py, then run utility/scripts/build_notebooks.py.

"""Lease locks that keep Lakehouse writers from overlapping.

``streaming-data-load`` (03, 04), ``daily-maintenance`` (05 and the ML
refresh), ``ml-required`` (setup-11 and the ML producers) and notebook 90 run
on independent schedules and write the same Delta tables. Fabric has no
cross-pipeline mutex, so each writer claims a lease row in
``<silver>._job_locks`` before it writes:

- ``silver``: notebook 03 and notebook 90 (all Silver tables and ``_watermarks``);
- ``gold``: notebook 04 and setup-11 (the ten Gold aggregates, ``setup_run_log``);
- ``ml:<table>``: the producer of that ML output table.

Notebook 05 claims the same lock for each table it maintains, without
waiting, and skips the table when another job holds it.

A claim is a Delta ``MERGE`` that succeeds only when the row is missing,
expired, or already ours; concurrent claims conflict in Delta and one of them
loses. The holder renews its lease from a background heartbeat, so a crashed
notebook releases its locks once the lease expires (``JOB_LOCK_TTL_SECONDS``).
All times come from Spark's ``current_timestamp()``, never the driver clock.

Keep this module free of package imports: Lakehouse notebooks run it as a
cell (``scripts/build_notebooks.py`` copies it in).
"""

import threading
import time
import uuid
from collections.abc import Callable, Iterable
from datetime import datetime, timedelta, timezone

JOB_LOCK_TABLE = "_job_locks"
JOB_LOCK_TTL_SECONDS = 900
JOB_LOCK_HEARTBEAT_SECONDS = 120
JOB_LOCK_POLL_SECONDS = 30
SILVER_LOCK = "silver"
GOLD_LOCK = "gold"


def ml_output_lock(table: str) -> str:
    """The lock that guards one ML output table."""
    return f"ml:{table}"


def is_concurrent_write_error(exc: BaseException) -> bool:
    """True for Delta optimistic-concurrency conflicts (another writer won)."""
    text = f"{type(exc).__name__}: {exc}"
    return "Concurrent" in text or "DELTA_CONCURRENT" in text


class DeltaJobLockStore:
    """Lease rows in a Delta table: (lock_name, owner, acquired_at, expires_at)."""

    def __init__(self, spark, table_name: str, ttl_seconds: int = JOB_LOCK_TTL_SECONDS):
        self.spark = spark
        self.table_name = table_name
        self.ttl_seconds = int(ttl_seconds)
        self.spark.sql(
            f"CREATE TABLE IF NOT EXISTS {table_name} "
            "(lock_name STRING, owner STRING, acquired_at TIMESTAMP, expires_at TIMESTAMP) "
            "USING DELTA"
        )

    @staticmethod
    def _quote(value: str) -> str:
        return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"

    def holder(self, name: str) -> str | None:
        """The owner of a live lease on ``name``, or None."""
        rows = self.spark.sql(
            f"SELECT owner FROM {self.table_name} "
            f"WHERE lock_name = {self._quote(name)} AND expires_at > current_timestamp()"
        ).collect()
        return rows[0]["owner"] if rows else None

    def try_claim(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"""
                MERGE INTO {self.table_name} AS t
                USING (
                    SELECT {self._quote(name)} AS lock_name, {self._quote(owner)} AS owner,
                           current_timestamp() AS acquired_at,
                           current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS AS expires_at
                ) AS s
                ON t.lock_name = s.lock_name
                WHEN MATCHED AND (t.expires_at <= s.acquired_at OR t.owner = s.owner)
                    THEN UPDATE SET t.owner = s.owner, t.acquired_at = s.acquired_at,
                                    t.expires_at = s.expires_at
                WHEN NOT MATCHED THEN INSERT (lock_name, owner, acquired_at, expires_at)
                    VALUES (s.lock_name, s.owner, s.acquired_at, s.expires_at)
                """
            )
        except Exception as exc:
            if is_concurrent_write_error(exc):
                return False
            raise
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"UPDATE {self.table_name} "
                f"SET expires_at = current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS "
                f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
            )
        except Exception as exc:
            if not is_concurrent_write_error(exc):
                raise
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        for _attempt in range(5):
            try:
                self.spark.sql(
                    f"DELETE FROM {self.table_name} "
                    f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
                )
                return
            except Exception as exc:
                if not is_concurrent_write_error(exc):
                    raise
                time.sleep(1)


class MemoryJobLockStore:
    """The same lease semantics in memory, with an injectable clock (tests)."""

    def __init__(
        self,
        ttl_seconds: int = JOB_LOCK_TTL_SECONDS,
        clock: Callable[[], datetime] | None = None,
    ):
        self.ttl = timedelta(seconds=ttl_seconds)
        self.clock = clock or (lambda: datetime.now(timezone.utc).replace(tzinfo=None))
        self.rows: dict[str, tuple[str, datetime]] = {}

    def holder(self, name: str) -> str | None:
        row = self.rows.get(name)
        return row[0] if row and row[1] > self.clock() else None

    def try_claim(self, name: str, owner: str) -> bool:
        now = self.clock()
        row = self.rows.get(name)
        if row is None or row[1] <= now or row[0] == owner:
            self.rows[name] = (owner, now + self.ttl)
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        row = self.rows.get(name)
        if row is not None and row[0] == owner:
            self.rows[name] = (owner, self.clock() + self.ttl)
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        if self.rows.get(name, ("", None))[0] == owner:
            del self.rows[name]


def new_lock_owner(holder: str) -> str:
    """A unique owner for one notebook run, e.g. ``03-streaming-to-silver:1a2b...``."""
    return f"{holder}:{uuid.uuid4().hex[:12]}"


def acquire_job_locks(
    store,
    names: Iterable[str],
    owner: str,
    *,
    wait_seconds: float,
    poll_seconds: float = JOB_LOCK_POLL_SECONDS,
    sleep: Callable[[float], None] = time.sleep,
    monotonic: Callable[[], float] = time.monotonic,
) -> list[str] | None:
    """Claim every lock in ``names`` or none of them.

    Locks are claimed in sorted order, so two jobs that need several locks
    can't deadlock. Retries until ``wait_seconds`` elapse (0 = one attempt)
    and returns the claimed names, or None when another job still holds one.
    """
    ordered = sorted(set(names))
    deadline = monotonic() + max(wait_seconds, 0)
    while True:
        claimed = []
        for name in ordered:
            if not store.try_claim(name, owner):
                break
            claimed.append(name)
        if len(claimed) == len(ordered):
            return claimed
        for name in reversed(claimed):
            store.release(name, owner)
        remaining = deadline - monotonic()
        if remaining <= 0:
            return None
        sleep(min(poll_seconds, remaining))


class JobLocks:
    """A notebook's set of leases, renewed by a heartbeat thread until released."""

    def __init__(
        self,
        store,
        holder: str,
        names: Iterable[str],
        heartbeat_seconds: float = JOB_LOCK_HEARTBEAT_SECONDS,
    ):
        self.store = store
        self.owner = new_lock_owner(holder)
        self.names = sorted(set(names))
        self.heartbeat_seconds = heartbeat_seconds
        self.held: list[str] = []
        self.lost: list[str] = []
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None

    def blockers(self) -> dict[str, str | None]:
        """Current holders of the locks this job needs (for messages)."""
        return {name: self.store.holder(name) for name in self.names}

    def acquire(self, wait_seconds: float, **kwargs) -> bool:
        claimed = acquire_job_locks(
            self.store, self.names, self.owner, wait_seconds=wait_seconds, **kwargs
        )
        if claimed is None:
            return False
        self.held = claimed
        if self.heartbeat_seconds > 0:
            self._thread = threading.Thread(target=self._heartbeat, daemon=True)
            self._thread.start()
        return True

    def _heartbeat(self) -> None:
        while not self._stop.wait(self.heartbeat_seconds):
            self.renew()

    def renew(self) -> None:
        for name in self.held:
            try:
                renewed = self.store.renew(name, self.owner)
            except Exception as exc:
                print(f"Lock {name}: renewal failed ({exc}); retrying at the next heartbeat")
                continue
            if not renewed and name not in self.lost:
                self.lost.append(name)

    def assert_held(self) -> None:
        """Fail before a commit point if a lease expired and another job took it."""
        if self.lost:
            raise RuntimeError(f"Job locks lost before completion: {self.lost}")

    def release(self) -> None:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=self.heartbeat_seconds + 5)
        for name in reversed(self.held):
            self.store.release(name, self.owner)
        self.held = []

In [ ]:
# One writer per ML output: this notebook runs in more than one pipeline, and
# notebook 05 maintains the same table (see job_locks).
LOCK_WAIT_SECONDS = int(get_env("LOCK_WAIT_SECONDS", default="1800"))
JOB_LOCKS = JobLocks(
    DeltaJobLockStore(spark, f"{LAKEHOUSE_NAME}.{SILVER_DB}.{JOB_LOCK_TABLE}"),
    "09-ml-churn-prediction",
    [ml_output_lock("churn_predictions")],
)
if not JOB_LOCKS.acquire(wait_seconds=LOCK_WAIT_SECONDS):
    raise RuntimeError(f"ML output locked by another job: {JOB_LOCKS.blockers()}")

In [ ]:
# =============================================================================
# HELPERS & MLFLOW SETUP
# =============================================================================


def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")


def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")


def silver_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False


def resolve_table_column(frame, table_name, *candidates):
    available = {column.lower(): column for column in frame.columns}
    for candidate in candidates:
        resolved = available.get(candidate.lower())
        if resolved is not None:
            return resolved
    raise RuntimeError(
        f"Unable to resolve any of {candidates} in {table_name}. "
        f"Available columns: {frame.columns}"
    )


def chronological_split_boundaries(
    ordered_dates,
    train_fraction=0.6,
    calibration_fraction=0.2,
    embargo_days=0,
):
    """Return chronological label-availability splits with horizon embargoes."""
    dates = sorted(set(ordered_dates))
    if len(dates) < 3:
        raise ValueError(
            "Chronological train/calibration/test requires at least 3 dates."
        )
    if embargo_days < 0:
        raise ValueError("Embargo days cannot be negative.")
    usable_span_days = (dates[-1] - dates[0]).days - (2 * embargo_days)
    if usable_span_days < 3:
        raise ValueError("Insufficient label-availability span after embargoes.")

    train_days = max(1, int(usable_span_days * train_fraction))
    calibration_days = max(1, int(usable_span_days * calibration_fraction))
    train_target = dates[0] + timedelta(days=train_days)
    train_end = max(date_value for date_value in dates if date_value <= train_target)
    calibration_start = next(
        (
            date_value
            for date_value in dates
            if date_value > train_end + timedelta(days=embargo_days)
        ),
        None,
    )
    if calibration_start is None:
        raise ValueError("No calibration dates remain after the training embargo.")
    calibration_target = calibration_start + timedelta(days=calibration_days)
    calibration_end = max(
        date_value
        for date_value in dates
        if calibration_start <= date_value <= calibration_target
    )
    test_start = next(
        (
            date_value
            for date_value in dates
            if date_value > calibration_end + timedelta(days=embargo_days)
        ),
        None,
    )
    if test_start is None:
        raise ValueError("No test dates remain after the calibration embargo.")
    return {
        "train_end": train_end,
        "calibration_start": calibration_start,
        "calibration_end": calibration_end,
        "test_start": test_start,
    }


def assert_unique_keys(frame, key_columns, context):
    duplicates = (
        frame.groupBy(*key_columns)
        .count()
        .filter(F.col("count") != 1)
        .limit(1)
        .count()
    )
    if duplicates:
        raise RuntimeError(f"{context} is not unique by {key_columns}.")


def require_binary_classes(frame, label_column, context):
    class_count = frame.select(label_column).distinct().count()
    if class_count != 2:
        raise RuntimeError(
            f"{context} must contain both classes for fail-closed training."
        )


def customer_activity(receipts, online_orders, as_of_timestamp):
    """Purchases of known customers up to the as-of: store SALE receipts and
    online orders. Anonymous purchases and returns never count."""
    def amount():
        return F.coalesce(
            F.col("total_amount").cast("double"),
            F.col("total_cents").cast("double") / 100.0,
        )

    store = receipts.filter(F.col("receipt_type") == "SALE").select(
        F.col("customer_id").cast("long").alias("customer_id"),
        F.col("event_ts").cast("timestamp").alias("event_ts"),
        F.col("store_id").cast("long").alias("store_id"),
        amount().alias("purchase_amount"),
        "payment_method",
        F.lit(0).alias("is_online"),
    )
    online = online_orders.select(
        F.col("customer_id").cast("long").alias("customer_id"),
        F.col("event_ts").cast("timestamp").alias("event_ts"),
        F.lit(None).cast("long").alias("store_id"),
        amount().alias("purchase_amount"),
        "payment_method",
        F.lit(1).alias("is_online"),
    )
    return (
        store.unionByName(online)
        .filter(
            F.col("customer_id").isNotNull()
            & (F.col("event_ts") <= F.lit(as_of_timestamp))
        )
        .withColumn("event_date", F.to_date("event_ts"))
    )


def daily_customer_activity(activity):
    """One row per customer and purchase day, aggregated before any join."""
    return activity.groupBy("customer_id", "event_date").agg(
        F.count(F.lit(1)).alias("purchases"),
        F.sum("is_online").alias("online_purchases"),
        F.sum("purchase_amount").alias("spend"),
        F.sum(F.col("purchase_amount") * F.col("purchase_amount")).alias("spend_sq"),
        F.max("purchase_amount").alias("max_basket"),
        F.min("purchase_amount").alias("min_basket"),
        F.collect_set("store_id").alias("stores"),
        F.collect_set("payment_method").alias("payment_methods"),
    )


def pair_with_snapshots(frame, snapshot_dates, after_days, through_days):
    """Pair each customer-day with the snapshots ``s`` for which
    ``after_days < datediff(s, event_date) <= through_days``; days pair only
    with the few snapshots whose window holds them (no cross join)."""
    dates = F.array(*[F.lit(value).cast("date") for value in sorted(snapshot_dates)])
    offset = lambda snapshot: F.datediff(snapshot, F.col("event_date"))  # noqa: E731
    return frame.withColumn(
        "snapshot_date",
        F.explode(
            F.filter(
                dates,
                lambda snapshot: (offset(snapshot) > after_days)
                & (offset(snapshot) <= through_days),
            )
        ),
    )


def build_feature_snapshots(daily_activity, customers, snapshot_dates):
    """Features from purchases in [snapshot - FEATURE_WINDOW_DAYS, snapshot),
    for customers who purchased in that window."""
    windowed = pair_with_snapshots(
        daily_activity, snapshot_dates, 0, FEATURE_WINDOW_DAYS
    )
    features = (
        windowed.groupBy("customer_id", "snapshot_date")
        .agg(
            F.sum("purchases").alias("purchase_count"),
            F.sum("online_purchases").alias("online_purchase_count"),
            F.sum("spend").alias("total_spend"),
            F.sum("spend_sq").alias("spend_sq"),
            F.max("max_basket").alias("max_basket"),
            F.min("min_basket").alias("min_basket"),
            F.size(F.array_distinct(F.flatten(F.collect_list("stores")))).alias(
                "unique_stores"
            ),
            F.size(
                F.array_distinct(F.flatten(F.collect_list("payment_methods")))
            ).alias("payment_methods_used"),
        )
        .withColumn("avg_basket_value", F.col("total_spend") / F.col("purchase_count"))
        .withColumn(
            "basket_std",
            F.when(
                F.col("purchase_count") > 1,
                F.sqrt(
                    F.greatest(
                        F.lit(0.0),
                        (
                            F.col("spend_sq")
                            - F.col("purchase_count")
                            * F.col("avg_basket_value")
                            * F.col("avg_basket_value")
                        )
                        / (F.col("purchase_count") - 1),
                    )
                ),
            ).otherwise(F.lit(0.0)),
        )
        .drop("spend_sq")
    )
    return (
        features.join(customers, on="customer_id", how="left")
        .withColumn(
            "online_share",
            F.col("online_purchase_count") / F.col("purchase_count"),
        )
        .withColumn(
            "purchase_frequency",
            F.col("purchase_count") / F.lit(float(FEATURE_WINDOW_DAYS)),
        )
        .withColumn(
            "basket_consistency",
            F.when(
                F.col("avg_basket_value") > 0,
                F.col("basket_std") / F.col("avg_basket_value"),
            ).otherwise(0.0),
        )
    )


def attach_forward_churn_labels(feature_snapshots, daily_activity, snapshot_dates):
    """Label inactivity only in [snapshot, snapshot + churn window)."""
    forward_active = (
        pair_with_snapshots(
            daily_activity.select("customer_id", "event_date"),
            snapshot_dates,
            -CHURN_WINDOW_DAYS,
            0,
        )
        .select("customer_id", "snapshot_date")
        .distinct()
        .withColumn("forward_active", F.lit(1))
    )
    return (
        feature_snapshots.join(
            forward_active, on=["customer_id", "snapshot_date"], how="left"
        )
        .withColumn(
            "is_churned",
            F.when(F.col("forward_active").isNull(), 1.0).otherwise(0.0),
        )
        .drop("forward_active")
        .withColumn(
            "label_available_date",
            F.date_add(F.col("snapshot_date"), CHURN_WINDOW_DAYS),
        )
    )


def raw_churn_scores(model, frame):
    return model.transform(frame).withColumn(
        "raw_churn_score",
        vector_to_array("probability")[1],
    )


def score_with_calibration(model, calibrator, calibration_assembler, frame):
    raw_scores = raw_churn_scores(model, frame)
    calibrated = calibrator.transform(
        calibration_assembler.transform(raw_scores)
    ).withColumn(
        "churn_probability",
        F.greatest(
            F.lit(0.0),
            F.least(F.lit(1.0), F.col("calibrated_probability")),
        ),
    )
    invalid_count = calibrated.filter(
        F.col("churn_probability").isNull()
        | (F.col("churn_probability") < 0.0)
        | (F.col("churn_probability") > 1.0)
    ).limit(1).count()
    if invalid_count:
        raise RuntimeError("Calibrated churn probabilities are invalid.")
    return calibrated.withColumn(
        "churn_prediction",
        (F.col("churn_probability") >= 0.5).cast("double"),
    )


ensure_database(GOLD_DB)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment '{EXPERIMENT_NAME}' ready")

def _normalize_ml_type(data_type):
    value = data_type.replace("bigint", "long").replace("integer", "int")
    return value


def validate_ml_output(frame, table_name):
    contract = ML_OUTPUT_CONTRACTS[table_name]
    expected = tuple((name, data_type) for name, data_type, _ in contract)
    actual = tuple(
        (field.name, _normalize_ml_type(field.dataType.simpleString()))
        for field in frame.schema.fields
    )
    if actual != expected:
        raise RuntimeError(
            f"ML output schema mismatch for {table_name}: expected={expected}, actual={actual}"
        )
    non_nullable = tuple(name for name, _, nullable in contract if not nullable)
    null_flags = frame.agg(
        *(F.max(F.col(name).isNull().cast("int")).alias(name) for name in non_nullable)
    ).first().asDict()
    null_columns = [name for name in non_nullable if null_flags.get(name)]
    if null_columns:
        raise RuntimeError(
            f"ML output {table_name} has nulls in required columns: {null_columns}"
        )
    return frame



## Data Validation & Preparation

In [ ]:
for table_name in [RECEIPTS_TABLE, ONLINE_ORDERS_TABLE, CUSTOMERS_TABLE]:
    if not silver_exists(table_name):
        raise RuntimeError(
            f"Required table {LAKEHOUSE_NAME}.{SILVER_DB}.{table_name} "
            "not found."
        )
    print(f"  {table_name}: OK")


In [ ]:
print("Preparing source-anchored historical snapshot dates...")

receipts_source = read_silver(RECEIPTS_TABLE)
# One source as-of for every required ML output: the latest Silver receipt.
source_range = receipts_source.agg(
    F.min(F.col("event_ts").cast("timestamp")).alias("first_source_ts"),
    F.max(F.col("event_ts").cast("timestamp")).alias("source_as_of_timestamp"),
).first()
source_as_of_timestamp = source_range["source_as_of_timestamp"]
if source_as_of_timestamp is None:
    raise RuntimeError(f"No transactions found in {RECEIPTS_TABLE}.")
source_start_date = source_range["first_source_ts"].date()
source_as_of_date = source_as_of_timestamp.date()
daily_activity_df = daily_customer_activity(
    customer_activity(
        receipts_source,
        read_silver(ONLINE_ORDERS_TABLE),
        source_as_of_timestamp,
    )
).cache()

customer_source = read_silver(CUSTOMERS_TABLE)
customer_id_column = resolve_table_column(
    customer_source, CUSTOMERS_TABLE, "customer_id", "id"
)
geography_id_column = resolve_table_column(
    customer_source, CUSTOMERS_TABLE, "geography_id", "geographyid"
)
customers_df = (
    customer_source.select(
        F.col(customer_id_column).cast("long").alias("customer_id"),
        F.col(geography_id_column).cast("double").alias("geography_id"),
    )
    .filter(F.col("customer_id").isNotNull())
    .groupBy("customer_id")
    .agg(F.min("geography_id").alias("geography_id"))
    .orderBy("customer_id")
    .cache()
)
assert_unique_keys(customers_df, ["customer_id"], "Customer dimension")

first_snapshot_date = source_start_date + timedelta(days=FEATURE_WINDOW_DAYS)
last_labeled_snapshot_date = source_as_of_date - timedelta(
    days=CHURN_WINDOW_DAYS
)
if first_snapshot_date > last_labeled_snapshot_date:
    raise RuntimeError(
        "Insufficient source history for pre-snapshot features and forward labels."
    )

snapshot_dates = [
    first_snapshot_date + timedelta(days=offset)
    for offset in range(
        0,
        (last_labeled_snapshot_date - first_snapshot_date).days + 1,
        SNAPSHOT_INTERVAL_DAYS,
    )
]
inference_snapshot_date = source_as_of_date + timedelta(days=1)

print(f"Source range: {source_range['first_source_ts']} to {source_as_of_timestamp}")
print(
    f"Labeled snapshots: {first_snapshot_date} to "
    f"{last_labeled_snapshot_date}"
)
print(f"Current inference boundary: {inference_snapshot_date}")


## Feature Engineering

Build behavioral features from purchase history and customer dimension
features from the available customer table columns. All operations use
Spark — fully distributed.

In [ ]:
print("Building strictly pre-snapshot behavioral features...")
historical_features = build_feature_snapshots(
    daily_activity_df,
    customers_df,
    snapshot_dates,
).cache()
current_features = build_feature_snapshots(
    daily_activity_df,
    customers_df,
    [inference_snapshot_date],
).cache()

assert_unique_keys(
    historical_features,
    ["customer_id", "snapshot_date"],
    "Historical churn features",
)
assert_unique_keys(
    current_features,
    ["customer_id"],
    "Current churn features",
)

print(f"Historical feature rows: {historical_features.count()}")
print(f"Current inference rows: {current_features.count()}")


In [ ]:
print("Attaching future-only churn labels...")
training_dataset = attach_forward_churn_labels(
    historical_features,
    daily_activity_df,
    snapshot_dates,
).cache()
assert_unique_keys(
    training_dataset,
    ["customer_id", "snapshot_date"],
    "Labeled churn snapshots",
)

ordered_label_available_dates = [
    row["label_available_date"]
    for row in training_dataset.select("label_available_date")
    .distinct()
    .orderBy("label_available_date")
    .collect()
]
split_dates = chronological_split_boundaries(
    ordered_label_available_dates,
    TRAIN_FRACTION,
    CALIBRATION_FRACTION,
    CHURN_WINDOW_DAYS,
)
train_label_available_end = split_dates["train_end"]
calibration_label_available_start = split_dates["calibration_start"]
calibration_label_available_end = split_dates["calibration_end"]
test_label_available_start = split_dates["test_start"]

print("Forward-label class distribution:")
training_dataset.groupBy("is_churned").count().orderBy("is_churned").show()
print(f"Train labels available through: {train_label_available_end}")
print(f"Calibration labels start after embargo: {calibration_label_available_start}")
print(f"Calibration labels available through: {calibration_label_available_end}")
print(f"Test labels available from: {test_label_available_start}")


## Model Training & Evaluation

Assemble features with `VectorAssembler`, train a distributed Spark ML
`GBTClassifier`, and evaluate with Spark evaluators — all tracked
in MLflow.

In [ ]:
feature_cols = [
    "geography_id",
    "purchase_count",
    "online_purchase_count",
    "online_share",
    "unique_stores",
    "total_spend",
    "avg_basket_value",
    "basket_std",
    "max_basket",
    "min_basket",
    "payment_methods_used",
    "purchase_frequency",
    "basket_consistency",
]


def prepare_model_frame(frame):
    prepared = frame
    for column_name in feature_cols:
        prepared = prepared.withColumn(
            column_name,
            F.col(column_name).cast("double"),
        )
    return prepared.na.fill(0.0, subset=feature_cols)


assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="error",
)
assembled_history = assembler.transform(
    prepare_model_frame(training_dataset)
)
df_train = assembled_history.filter(
    F.col("label_available_date") <= F.lit(train_label_available_end)
)
df_calibration = assembled_history.filter(
    (F.col("label_available_date") >= F.lit(calibration_label_available_start))
    & (
        F.col("label_available_date")
        <= F.lit(calibration_label_available_end)
    )
)
df_test = assembled_history.filter(
    F.col("label_available_date") >= F.lit(test_label_available_start)
)

for split_name, split_frame in [
    ("Training split", df_train),
    ("Calibration split", df_calibration),
    ("Test split", df_test),
]:
    if split_frame.count() == 0:
        raise RuntimeError(f"{split_name} is empty.")
    require_binary_classes(split_frame, "is_churned", split_name)

train_count = df_train.count()
calibration_count = df_calibration.count()
test_count = df_test.count()
print(
    f"Chronological rows -- train: {train_count}, "
    f"calibration: {calibration_count}, test: {test_count}"
)

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="is_churned",
    maxIter=MAX_ITER,
    maxDepth=MAX_DEPTH,
    stepSize=STEP_SIZE,
    subsamplingRate=SUBSAMPLING_RATE,
    seed=RANDOM_SEED,
)

with mlflow.start_run(
    run_name=f"gbt_churn_{source_as_of_timestamp.strftime('%Y%m%d_%H%M')}"
) as run:
    model = gbt.fit(df_train.select("features", "is_churned"))

    raw_calibration = raw_churn_scores(model, df_calibration).cache()
    if raw_calibration.select("raw_churn_score").distinct().count() < 2:
        raise RuntimeError(
            "Churn calibration has fewer than two distinct model scores."
        )
    calibration_assembler = VectorAssembler(
        inputCols=["raw_churn_score"],
        outputCol="calibration_features",
        handleInvalid="error",
    )
    calibrator = IsotonicRegression(
        featuresCol="calibration_features",
        labelCol="is_churned",
        predictionCol="calibrated_probability",
        isotonic=True,
    ).fit(calibration_assembler.transform(raw_calibration))

    df_predictions = score_with_calibration(
        model,
        calibrator,
        calibration_assembler,
        df_test,
    )
    binary_evaluator = BinaryClassificationEvaluator(
        labelCol="is_churned",
        rawPredictionCol="churn_probability",
        metricName="areaUnderROC",
    )
    auc_roc = float(binary_evaluator.evaluate(df_predictions))
    multiclass_evaluator = MulticlassClassificationEvaluator(
        labelCol="is_churned",
        predictionCol="churn_prediction",
    )
    precision = float(
        multiclass_evaluator.evaluate(
            df_predictions,
            {multiclass_evaluator.metricName: "weightedPrecision"},
        )
    )
    recall = float(
        multiclass_evaluator.evaluate(
            df_predictions,
            {multiclass_evaluator.metricName: "weightedRecall"},
        )
    )
    f1 = float(
        multiclass_evaluator.evaluate(
            df_predictions,
            {multiclass_evaluator.metricName: "f1"},
        )
    )
    brier_score = float(
        df_predictions.agg(
            F.avg(
                F.pow(
                    F.col("churn_probability") - F.col("is_churned"),
                    2,
                )
            ).alias("brier_score")
        ).first()["brier_score"]
    )

    churn_rate = float(
        df_train.agg(F.avg("is_churned").alias("churn_rate")).first()[
            "churn_rate"
        ]
    )
    mlflow.log_params({
        "algorithm": "gbt_classifier_isotonic_calibration",
        "churn_window_days": CHURN_WINDOW_DAYS,
        "feature_window_days": FEATURE_WINDOW_DAYS,
        "snapshot_interval_days": SNAPSHOT_INTERVAL_DAYS,
        "max_iter": MAX_ITER,
        "max_depth": MAX_DEPTH,
        "step_size": STEP_SIZE,
        "subsampling_rate": SUBSAMPLING_RATE,
        "feature_count": len(feature_cols),
        "train_rows": train_count,
        "calibration_rows": calibration_count,
        "test_rows": test_count,
        "source_as_of": source_as_of_timestamp.isoformat(),
    })
    mlflow.log_metrics({
        "calibrated_auc_roc": round(auc_roc, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "brier_score": round(brier_score, 6),
        "train_churn_rate": round(churn_rate, 4),
    })
    mlflow.spark.log_model(model, "gbt_churn_model")
    mlflow.spark.log_model(calibrator, "isotonic_churn_calibrator")

    print(f"MLflow run: {run.info.run_id}")
    print(f"Calibrated test AUC-ROC: {auc_roc:.4f}")
    print(f"Precision: {precision:.4f}; Recall: {recall:.4f}; F1: {f1:.4f}")
    print(f"Brier score: {brier_score:.6f}")


## Save Predictions to Gold Layer

Score the full dataset and write predictions to Delta.

In [ ]:
print("Scoring one current inference snapshot per customer...")
current_assembled = assembler.transform(prepare_model_frame(current_features))
df_scored = score_with_calibration(
    model,
    calibrator,
    calibration_assembler,
    current_assembled,
)
assert_unique_keys(df_scored, ["customer_id"], "Current churn scores")

df_scored = df_scored.withColumn(
    "risk_category",
    F.when(F.col("churn_probability") < 0.2, "Very Low")
    .when(F.col("churn_probability") < 0.4, "Low")
    .when(F.col("churn_probability") < 0.6, "Medium")
    .when(F.col("churn_probability") < 0.8, "High")
    .otherwise("Very High"),
)

model_version = f"gbt_iso_{run.info.run_id[:8]}"
df_final = (
    df_scored.select(
        F.col("customer_id").cast("long"),
        F.col("churn_probability").cast("double"),
        F.col("churn_prediction").cast("int"),
        F.col("risk_category").cast("string"),
        F.lit(None).cast("int").alias("is_churned_actual"),
        F.lit(source_as_of_timestamp)
        .cast("timestamp")
        .alias("prediction_date"),
        F.current_timestamp().alias("generated_at"),
        F.lit(model_version).cast("string").alias("model_version"),
        F.lit(CHURN_WINDOW_DAYS)
        .cast("int")
        .alias("churn_window_days"),
        F.lit(run.info.run_id).cast("string").alias("model_run_id"),
        F.lit("1.0").cast("string").alias("schema_version"),
    )
    .orderBy("customer_id")
)
assert_unique_keys(df_final, ["customer_id"], "Churn inference output")

df_final = validate_ml_output(df_final, "churn_predictions")
df_final.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(
    CHURN_PREDICTIONS_TABLE_NAME
)
saved_count = spark.table(CHURN_PREDICTIONS_TABLE_NAME).count()
if saved_count != current_features.count():
    raise RuntimeError(
        "Published churn output does not contain exactly one row per active customer."
    )
print(f"Saved {saved_count} current predictions to {CHURN_PREDICTIONS_TABLE_NAME}")

risk_distribution = (
    spark.table(CHURN_PREDICTIONS_TABLE_NAME)
    .groupBy("risk_category")
    .agg(F.count("*").alias("count"))
    .orderBy("risk_category")
    .collect()
)
for row in risk_distribution:
    percentage = row["count"] / saved_count * 100.0
    print(
        f"  {row['risk_category']:<12} {row['count']:>6} "
        f"({percentage:>5.1f}%)"
    )


In [ ]:
print("=" * 60)
print("CHURN PREDICTION COMPLETE")
print("=" * 60)
print(f"Prediction table: {CHURN_PREDICTIONS_TABLE_NAME}")
print(f"Source as-of: {source_as_of_timestamp}")
print(f"Forward churn window: {CHURN_WINDOW_DAYS} days")
print("Published probabilities: isotonic calibrated")
print("Published grain: one current row per active customer; no actual label")
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(
    f"Schedule this notebook on a {SCHEDULE_CADENCE} cadence for fresh "
    "predictions."
)


In [ ]:
# Release the ML output lock (see job_locks).
JOB_LOCKS.assert_held()
JOB_LOCKS.release()